In [20]:
# Aggregation Agent 34 picks up where 32 left off in revised edits.
# Aggregation Agent 34 is built off of Aggregation Agent 33.

In [21]:
# I believe that the modifications in iterating on batch sizes comes from
# chunk_stream() # Section 2
# build_activity_candidates() # Section 2B
# 

In [ ]:
#Edited Code Test

In [ ]:
import pandas as pd
import fitz, re, json, os, gzip, gc, shutil, time
from typing import Dict, List, Tuple
from openai import AzureOpenAI
from azure.identity import DeviceCodeCredential, get_bearer_token_provider
# NEW: for embedding-based retrieval (local FAISS example)
import numpy as np
import faiss

from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.style import WD_STYLE_TYPE
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.shared import Pt, Inches, RGBColor
from collections import defaultdict, deque

import matplotlib.pyplot as plt
from io import BytesIO

from pathlib import Path

# =========================
# 1) PATHS & INPUTS
# =========================
PDF_PATH   = r"C:/Users/wb643925/OneDrive - WBG/SmartProcurement/Procurement-Regulations-7th-Edition-Sep-2025.pdf"
LOCAL_OUT  = r"C:/Users/wb643925/OneDrive - WBG/SmartProcurement/tmp/wb_procurement_chunks.jsonl.gz"
DBFS_OUT   = r"C:/Users/wb643925/OneDrive - WBG/SmartProcurement/wb_procurement_chunks.jsonl.gz"
META_PATH  = r"C:/Users/wb643925/OneDrive - WBG/SmartProcurement/wb_procurement_chunks.meta.json"

CSV_ACTIVITIES = r"C:/Users/wb643925/OneDrive - WBG/SmartProcurement/merged.csv" # from Data Explorer
XLSX_SCHEMA    = r"C:/Users/wb643925/OneDrive - WBG/SmartProcurement/Grid_Data.xlsx"
SCHEMA_JSON    = r"C:/Users/wb643925/OneDrive - WBG/SmartProcurement/procurement_schema.json"

TARGET_PROJ_ID = "P166309" # "P173671"
REPORT_DIR = r"C:/Users/wb643925/OneDrive - WBG/SmartProcurement/"
-
# Make sure the tmp folder exists
os.makedirs(os.path.dirname(LOCAL_OUT), exist_ok=True)

In [23]:
# Use "P164920" and "P166309" to see if I iterated batch sizing correctly

In [24]:
# 👈 set your target project here  # P128012 # P145162 # P143843 # P174259 # "P169548" # P176114 # P173671 # P180811 # "P164920" # "P166309"
# For reasons I do not understand, P173671 and P180811 are broken.
# Aggregation Agents works with other projects, but not P173671 and P180811.

In [25]:
# =========================
# 2) PDF CLEAN/CHUNK
# =========================
_hdr_pat   = re.compile(r"^\s*(World Bank|Procurement Regulations|[A-Z].*Regulations).*$", re.I)
_page_no   = re.compile(r"^\s*\d+\s*$")
multispace = re.compile(r"[ \t]+")
hyphen_ln  = re.compile(r"-\n(?=[a-z])", re.I)

def clean_lines(lines: List[str]) -> str:
    kept = []
    for ln in lines:
        if _hdr_pat.match(ln): 
            continue
        if _page_no.match(ln): 
            continue
        kept.append(multispace.sub(" ", ln).rstrip())
    text = "\n".join(kept).strip()
    text = hyphen_ln.sub("", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text

def build_page_section_map(doc: fitz.Document) -> List[Tuple[int, int, str]]:
    toc = doc.get_toc(simple=True)
    spans = []
    for i, (_, title, start) in enumerate(toc):
        start = max(1, start)
        end = doc.page_count if i == len(toc)-1 else max(1, toc[i+1][2]-1)
        spans.append((start, end, title))
    return spans

def section_for_page(spans, page_no: int) -> str:
    for s, e, t in spans:
        if s <= page_no <= e: 
            return t
    return ""

def extract_page_text(page: fitz.Page) -> str:
    blocks = page.get_text("blocks")
    blocks = sorted(blocks, key=lambda b: (round(b[1], 1), round(b[0], 1)))
    lines = []
    for b in blocks:
        txt = (b[4] or "").splitlines()
        lines.extend(txt)
    return clean_lines(lines)

def chunk_stream(text: str, page_no: int, section: str,
                max_chars: int = 8000, overlap: int = 400, edition: str = "Seventh Edition, Sep 2025"):
    start = 0
    L = len(text)
    while start < L:
        end = min(L, start + max_chars)
        yield {
            "page": page_no,
            "section": section,
            "edition": edition,
            "content": text[start:end].strip()
        }
        if end == L: 
            break
        start = end - overlap

In [26]:
# =========================
# 2A) AUTH TO MAI (Azure OpenAI)
# =========================
token_provider = get_bearer_token_provider(
    DeviceCodeCredential(
        tenant_id="31a2fec0-266b-4c67-b56e-2796d8f59c36",
        client_id="00c104af-b0ae-4557-9787-6e6cfced741e"
    ),
    "https://cognitiveservices.azure.com/.default"
)

client = AzureOpenAI(
    azure_endpoint="https://azapimdev.worldbank.org/conversationalai/v2",
    azure_ad_token_provider=token_provider,
    api_version="2025-01-01-preview"
)

In [27]:
# =========================
# 2B) BUILD / LOAD VECTOR INDEX FOR POLICY CHUNKS (ONE-TIME / REUSABLE)
# =========================
EMBED_INDEX_PATH = r"C:/Users/wb643925/OneDrive - WBG/SmartProcurement/wb_policy_index.faiss"
EMBED_META_PATH  = r"C:/Users/wb643925/OneDrive - WBG/SmartProcurement/wb_policy_index_meta.json"
EMBED_MODEL_NAME = "text-embedding-3-large"

def embed_texts(texts: List[str]) -> np.ndarray:
    """
    Call Azure OpenAI embeddings endpoint to get vector representations.
    """
    emb_resp = client.embeddings.create(
        model=EMBED_MODEL_NAME,
        input=texts,
    )
    vectors = [d.embedding for d in emb_resp.data]
    return np.array(vectors, dtype="float32")

def build_activity_candidates(
    activities: List[dict],
    embed_fn,
    k: int = 20,
    sim_threshold: float = 0.35,
    max_desc_chars: int = 800,
    strategy: str = "top_clusters",          # NEW: "top_clusters" or "components"
    max_groups_per_cat: int = 20,   # 30
    max_group_size: int = 20,        # 25
    top_pairs_per_cat: int = 250             # NEW: controls group count pressure
) -> Dict[str, List[List[str]]]:
    by_cat = defaultdict(list)
    for a in activities:
        cat = a.get("procurement category", "") or "Unknown"
        by_cat[cat].append(a)

    cat_to_groups = {}
    for cat, items in by_cat.items():
        if len(items) < 2:
            cat_to_groups[cat] = []
            continue

        # Keep embeddings fast by truncating more aggressively (descriptions are often verbose)
        texts = [(it.get("description", "") or "")[:max_desc_chars] for it in items]
        vecs = embed_fn(texts).astype("float32")
        faiss.normalize_L2(vecs)

        dim = vecs.shape[1]
        index = faiss.IndexFlatIP(dim)
        index.add(vecs)

        k_eff = min(k + 1, len(items))
        D, I = index.search(vecs, k_eff)

        adj = defaultdict(set)
        # for i in range(len(items)):
        #     for jpos in range(1, k_eff):
        #         j = int(I[i, jpos])
        #         sim = float(D[i, jpos])
        #         if sim < sim_threshold:
        #             continue
        #         adj[i].add(j)
        #         adj[j].add(i)

        # seen = set()
        # groups = []
        # for i in range(len(items)):
        #     if i in seen:
        #         continue
        #     q = deque([i])
        #     comp = []
        #     seen.add(i)
        #     while q:
        #         u = q.popleft()
        #         comp.append(u)
        #         for v in adj[u]:
        #             if v not in seen:
        #                 seen.add(v)
        #                 q.append(v)

        #     if len(comp) >= 2:
        #         # Cap group size to avoid huge payloads + huge LLM costs
        #         comp = comp[:max_group_size]
        #         groups.append([items[idx]["activity_id"] for idx in comp])

        #         # Cap number of groups per category 
        #         if len(groups) >= max_groups_per_category:
        #             break

        # cat_to_groups[cat] = groups
                # Build list of candidate pairs above threshold
        pairs = []
        for i in range(len(items)):
            for jpos in range(1, k_eff):
                j = int(I[i, jpos])
                if j == i:
                    continue
                sim = float(D[i, jpos])
                if sim < sim_threshold:
                    continue
                a, b = (i, j) if i < j else (j, i)
                pairs.append((sim, a, b))

        # De-duplicate pairs and keep only strongest instances
        # (same pair can appear multiple times across neighbors)
        best = {}
        for sim, a, b in pairs:
            key = (a, b)
            if key not in best or sim > best[key]:
                best[key] = sim
        pairs = [(sim, a, b) for (a, b), sim in best.items()]
        pairs.sort(reverse=True, key=lambda x: x[0])

        if strategy == "components":
            # Original behavior: connected components on adjacency graph
            adj = defaultdict(set)
            for sim, a, b in pairs:
                adj[a].add(b)
                adj[b].add(a)

            seen = set()
            groups = []
            for i in range(len(items)):
                if i in seen:
                    continue
                q = deque([i])
                comp = []
                seen.add(i)
                while q:
                    u = q.popleft()
                    comp.append(u)
                    for v in adj[u]:
                        if v not in seen:
                            seen.add(v)
                            q.append(v)

                if len(comp) >= 2:
                    comp = comp[:max_group_size]
                    groups.append([items[idx]["activity_id"] for idx in comp])
                    if len(groups) >= max_groups_per_cat:
                        break

            cat_to_groups[cat] = groups

        else:
            # NEW behavior: greedy top-clusters only (non-overlapping)
            # Take top pairs, turn into clusters, avoid chaining explosion.
            pairs = pairs[:top_pairs_per_cat]

            assigned = set()
            groups = []

            # Precompute neighbor lists for fast expansion
            neigh = defaultdict(list)
            for sim, a, b in pairs:
                neigh[a].append((sim, b))
                neigh[b].append((sim, a))

            for sim, a, b in pairs:
                if a in assigned or b in assigned:
                    continue

                # Start a cluster from a strong pair
                cluster = [a, b]
                assigned.add(a)
                assigned.add(b)

                # Expand cluster using strongest neighbors of current members
                # (still respects caps so it won't balloon)
                candidates = []
                for u in cluster:
                    for sim2, v in neigh.get(u, []):
                        if v in assigned:
                            continue
                        candidates.append((sim2, v))

                candidates.sort(reverse=True, key=lambda x: x[0])

                for sim2, v in candidates:
                    if len(cluster) >= max_group_size:
                        break
                    if v in assigned:
                        continue
                    cluster.append(v)
                    assigned.add(v)

                if len(cluster) >= 2:
                    groups.append([items[idx]["activity_id"] for idx in cluster])

                if len(groups) >= max_groups_per_cat:
                    break

            cat_to_groups[cat] = groups


    return cat_to_groups   

def build_policy_index(chunks_path: str, rebuild: bool = False):
    """
    Read all policy chunks from wb_procurement_chunks.jsonl.gz,
    embed them, and build a FAISS index for later retrieval.
    Stores:
      - FAISS index file at EMBED_INDEX_PATH
      - metadata (list of {id, section, page, content}) at EMBED_META_PATH
    """
    if os.path.exists(EMBED_INDEX_PATH) and os.path.exists(EMBED_META_PATH) and not rebuild:
        print("✅ Existing policy index found; skipping rebuild.")
        return

    print("🔧 Building policy embedding index...")
    docs = []
    with gzip.open(chunks_path, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            docs.append({
                "id": i,
                "section": obj.get("section", ""),
                "page": obj.get("page", ""),
                "content": obj.get("content", ""),
            })

    # Embed in small batches to stay within per-request token limits
    batch_size = 8                      # ↓ was 64
    MAX_CHARS_PER_EMBED = 2000          # safety cap per chunk

    all_vectors = []
    for start in range(0, len(docs), batch_size):
        batch = docs[start:start+batch_size]
        # Truncate each chunk so no single text is huge
        texts = [d["content"][:MAX_CHARS_PER_EMBED] for d in batch]

        vecs = embed_texts(texts)
        all_vectors.append(vecs)
        print(f"  Embedded {start + len(batch)} / {len(docs)} chunks")

    if not all_vectors:
        raise ValueError("No valid chunks to index.")

    all_vectors = np.vstack(all_vectors)

    # Build FAISS index
    dim = all_vectors.shape[1]
    index = faiss.IndexFlatIP(dim)
    faiss.normalize_L2(all_vectors)
    index.add(all_vectors)

    faiss.write_index(index, EMBED_INDEX_PATH)
    with open(EMBED_META_PATH, "w", encoding="utf-8") as f:
        json.dump(docs, f, ensure_ascii=False, indent=2)

    print(f"✅ Policy index built with {len(docs)} chunks.")    

def process_pdf_fast(pdf_path: str,
                     out_jsonl_gz_local: str,
                     max_chars: int = 8000,
                     overlap: int = 400,
                     flush_every_pages: int = 25,
                     resume_from_page: int = 1) -> Dict:
    if os.path.exists(out_jsonl_gz_local) and resume_from_page == 1:
        os.remove(out_jsonl_gz_local)

    t0 = time.time()
    total_pages = 0
    total_chunks = 0

    doc = fitz.open(pdf_path)
    spans = build_page_section_map(doc)

    with gzip.open(out_jsonl_gz_local, "ab") as gz:
        for pno in range(resume_from_page, doc.page_count + 1):
            page = doc.load_page(pno - 1)
            text = extract_page_text(page)
            sect = section_for_page(spans, pno)

            if len(text) < 20:
                total_pages += 1
                continue

            for ch in chunk_stream(text, pno, sect, max_chars=max_chars, overlap=overlap):
                gz.write((json.dumps(ch, ensure_ascii=False) + "\n").encode("utf-8"))
                total_chunks += 1

            total_pages += 1
            if pno % flush_every_pages == 0:
                gz.flush()
                gc.collect()
                print(f"✅ {pno}/{doc.page_count} pages processed (chunks: {total_chunks})")

    doc.close()
    dt = time.time() - t0
    return {"pages": total_pages, "chunks": total_chunks, "seconds": round(dt, 2), "local_out": out_jsonl_gz_local}   

# Run PDF processing ONLY if chunks file is missing
if not os.path.exists(DBFS_OUT):
    meta = process_pdf_fast(PDF_PATH, LOCAL_OUT)
    if os.path.exists(LOCAL_OUT):
        if os.path.exists(DBFS_OUT):
            os.remove(DBFS_OUT)
        shutil.move(LOCAL_OUT, DBFS_OUT)
        print(f"Moved {LOCAL_OUT} → {DBFS_OUT}")
    else:
        raise FileNotFoundError(f"Expected {LOCAL_OUT} but it was not created.")

    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump({"pdf": PDF_PATH, "out": DBFS_OUT, **meta}, f, ensure_ascii=False, indent=2)
    print("Done:", {"pdf": PDF_PATH, "out": DBFS_OUT, **meta})
else:
    print(f"✅ Found existing policy chunks at {DBFS_OUT}; skipping PDF chunking.")

✅ Found existing policy chunks at C:/Users/wb643925/OneDrive - WBG/SmartProcurement/wb_procurement_chunks.jsonl.gz; skipping PDF chunking.


In [28]:
# =========================
# 2D) USE POLICY CHUNKS IN MODEL
# =========================
def _load_policy_index_and_meta():
    if not (os.path.exists(EMBED_INDEX_PATH) and os.path.exists(EMBED_META_PATH)):
        raise FileNotFoundError("Policy index or metadata not found. Run build_policy_index(...) first.")

    index = faiss.read_index(EMBED_INDEX_PATH)
    with open(EMBED_META_PATH, "r", encoding="utf-8") as f:
        meta = json.load(f)
    return index, meta

def build_policy_query_from_plan(user_payload: dict) -> str:
    project_id = user_payload.get("project_id", "")
    methods = []
    categories = []
    for act in user_payload.get("activities", []):
        m = act.get("method", "")
        c = act.get("procurement category", "")
        if m:
            methods.append(m)
        if c:
            categories.append(c)

    methods_str = ", ".join(sorted(set(methods)))
    cats_str = ", ".join(sorted(set(categories)))

    return (
        f"World Bank procurement regulations for planning, methods, thresholds, "
        f"review types, aggregation and splitting, and rated criteria. "
        f"Project ID: {project_id}. "
        f"Methods used: {methods_str}. "
        f"Categories: {cats_str}."
    )

def retrieve_policy_chunks_for_plan(user_payload: dict, top_k: int = 15) -> str:
    """
    Use embedding-based retrieval against the Regulations index to get the most relevant chunks
    for the current procurement plan.
    Returns a concatenated string of [Section | Page] + content.
    """
    index, meta = _load_policy_index_and_meta()

    query_text = build_policy_query_from_plan(user_payload)
    query_vec = embed_texts([query_text]).astype("float32")
    faiss.normalize_L2(query_vec)

    D, I = index.search(query_vec, top_k)
    # I[0] is the list of indices of top_k chunks
    selected = []
    for idx in I[0]:
        doc = meta[int(idx)]
        selected.append(
            f"[Section: {doc.get('section', '')} | Page: {doc.get('page', '')}]\n"
            f"{doc.get('content', '')}"
        )

    if not selected:
        return "No relevant regulations chunks could be retrieved; policy reasoning may be incomplete."

    policy_context = "\n\n---\n\n".join(selected)
    return policy_context[:16000]

In [29]:
# =========================
# 3) LOAD DATA & BUILD SCHEMA
# =========================
df = pd.read_csv(CSV_ACTIVITIES, encoding='ISO-8859-1', low_memory=False)
df_descriptions = pd.read_excel(XLSX_SCHEMA)
df_descriptions.columns = [c.strip() for c in df_descriptions.columns]

schema = {}
for _, row in df_descriptions.iterrows():
    technical = str(row.get("Technical Name", "")).strip()
    data_type = str(row.get("Data Type", "")).strip()
    desc = str(row.get("Description", "")).strip()
    if technical:
        schema[technical] = {"type": data_type, "desc": desc}

with open(SCHEMA_JSON, "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)
print(f"✅ Schema saved to {SCHEMA_JSON}")

✅ Schema saved to C:/Users/wb643925/OneDrive - WBG/SmartProcurement/procurement_schema.json


In [30]:
# Khalid: Only generate improvements and check compliance for activities which are 'Under Implementation', 'Signed', 'Pending Implementation', 'Submitted', 'Under Review','Uncanceled'
# keep_activities = ['Under Implementation', 'Signed', 'Pending Implementation', 'Submitted', 'Under Review','Uncanceled']
# df = df[df['PROC_STAT_DESC'].isin(keep_activities)]

In [31]:
# =========================
# 4) BUILD POLICY INDEX
# =========================
if not os.path.exists(DBFS_OUT):
    raise FileNotFoundError(f"Policy chunks file not found at {DBFS_OUT}. PDF chunking must run first.")
build_policy_index(DBFS_OUT, rebuild=False)

✅ Existing policy index found; skipping rebuild.


In [32]:
# THE SECTION ABOVE IS DIFFERENT COMPARED TO AGGREGATION AGENT 18

In [33]:
# =========================
# 5) BUILD FULL-PLAN PAYLOAD
# =========================
def _to_str(x):
    return "" if pd.isna(x) else str(x)

def _to_float(x):
    try:
        return float(x) if pd.notna(x) else 0.0
    except:
        return 0.0

df_project = df[df["PROJ_ID"] == TARGET_PROJ_ID].copy()

activities_payload = []
approach_map = {"O": "Open approach", "L": "Limited approach", "D": "Direct approach"}

for _, r in df_project.iterrows():
    raw_approach = _to_str(r.get("MRKT_APRCH_CODE"))
    mapped_approach = approach_map.get(raw_approach, "Unknown")

    activities_payload.append({
        "activity_id": _to_str(r.get("ACTVTY_ID")),
        "activity_name": _to_str(r.get("ACTV_DESC", "N/A")),
        "description": _to_str(r.get("ACTV_DESC", "N/A")),
        "procurement category": _to_str(r.get("PROC_GRP_DESC", "Unknown")),
        "method": _to_str(r.get("PROC_METH_CODE", "Unknown")),
        "approach": mapped_approach,
        "estimated_cost_usd": _to_float(r.get("EST_BUDG_AMT")),
        "planned_dates": {
            "activity_start_date": _to_str(r.get("Start")),
            "activity_end_date": _to_str(r.get("End"))
        }
    })

canonical_activity_ids = [a["activity_id"] for a in activities_payload]
canonical_activity_count = len(canonical_activity_ids)
print("Canonical activity count:", canonical_activity_count)

user_payload = {
    "user_type": "Borrower",
    "project_id": TARGET_PROJ_ID,
    "activities": activities_payload
}

Canonical activity count: 345


In [34]:
# =========================
# 6) SYSTEM PROMPT (unrevised)
# =========================
system_prompt_agg_ag = """You are the **World Bank Procurement Planning Advisor**, an AI agent that reviews procurement plans under World Bank–financed projects. \
Your goal is to analyze activities for potential aggregation opportunities to improve procurement efficiency and compliance. \

Focus strictly on the **planning phase**. \
Align every analysis with the **World Bank Procurement Regulations**, **Directives**, **Country thresholds**, and **March 2025 Procurement Change Management priorities** (aggregation). \

You must detect and analyze the following dimensions for each procurement activity: \
1️⃣ Aggregation and splitting activities (within project and across country portfolio) \

CANONICAL ACTIVITY LIST AND COUNTS
- The ONLY activities you may use are those in user_payload["activities"].
- Let N = len(user_payload["activities"]).
- In the executive_summary, "total_activities" MUST equal N.
- The scored_dashboard MUST contain exactly one row per activity in user_payload["activities"].
- The modified_plan MUST contain exactly one element per activity_id in user_payload["activities"].
- You MUST NOT create or delete activities. Aggregation is represented only by the "aggregation_group" field; it does NOT change how many activities exist.
- Any count of "all activities" MUST equal N unless you are explicitly counting a subset (e.g., only non-compliant). In that case, the subset + its complement MUST sum to N.

**MANDATORY INSTRUCTIONS FOR HOW TO FORMAT RESPONSES**: \
*For every recommendation, explicitly reference the procurement activity (PROC_GRP_DESC).
When suggesting aggregation, list all activity descriptions (ACTV_DESC) to be aggregated together in *BOTH* the Activity Descriptions column *AND* the Description of Improvement(s) column in the Aggregation Table Excel file. If the activities are to be aggregated into more than one group, specify which activities to group together and into how many groups. \

**MANDATORY INSTRUCTIONS FOR EVERY AGGREGATION RECOMMENDATION (each one must be followed):** \
Two activities can *ONLY* be aggregated if they have the *SAME* procurement category (PROC_GRP_DESC). \
Two activities with *DIFFERENT* procurement categories (PROC_GRP_DESC) **CANNOT** be aggregated.\
Activities with the procurement method code of "INDV" (PROC_METH_CODE) must NOT be considered for aggregation. **\

Always return JSON matching the provided schema. Never output conversational filler.
"""

In [35]:
# THE SECTION ABOVE IS DIFFERENT COMPARED TO AGGREGATION AGENT 18

In [36]:
# =========================
# 6B) RESPONSE FORMAT FULL
# =========================
response_format_full = {
    "type": "json_schema",
    "json_schema": {
        "name": "generate_procurement_review",
        "description": "Analyzes a procurement plan under World Bank–financed projects and generates diagnostics, dashboard, summary, improvement table, and modified plan.",
        "schema": {
            "type": "object",
            "properties": {
                "executive_summary": {
                    "type": "object",
                    "properties": {
                        "total_activities": {"type": "integer"},
                        "high_risk": {"type": "integer"},
                        "medium_risk": {"type": "integer"},
                        "low_risk": {"type": "integer"},
                        "top_recommendations": {"type": "array", "items": {"type": "string"}},
                        "expected_impact": {"type": "string"}
                    }
                },
                "findings_by_class": {"type": "array", "items": {"type": "object"}},
                "scored_dashboard": {"type": "array", "items": {"type": "object"}},
                "policy_alignment": {"type": "array", "items": {"type": "string"}},
                "improvement_table": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "Type of Improvement": {"type": "string"},
                            "Activity IDs": {"type": "string"},
                            "Activity Descriptions": {"type": "string"},
                            "Sum of Costs": {"type": "string"},
                            "Number of Activities": {"type": "integer"},
                            "Description of Improvement(s)": {"type": "string"},
                            "Achievement of Aggregation": {"type": "string"},
                            "Form of Aggregation": {"type": "string"},
                            "Considerations for Aggregation": {"type": "string"}
                        },
                        "required": [
                            "Type of Improvement",
                            "Activity IDs",
                            "Activity Descriptions",
                            "Sum of Costs",
                            "Number of Activities",
                            "Description of Improvement(s)",
                            "Achievement of Aggregation",
                            "Form of Aggregation",
                            "Considerations for Aggregation"
                        ]
                    }
                },
                "modified_plan": {"type": "array", "items": {"type": "object"}}
            },
            "required": ["executive_summary", "improvement_table"]
        }
    }
}

In [37]:
# =========================
# 6C) RESPONSE FORMAT AGG
# =========================
response_format_agg_only = {
    "type": "json_schema",
    "json_schema": {
        "name": "generate_aggregation_only",
        "description": "Returns only aggregation improvement_table rows for the subset.",
        "schema": {
            "type": "object",
            "properties": {
                "improvement_table": response_format_full["json_schema"]["schema"]["properties"]["improvement_table"]
            },
            "required": ["improvement_table"]
        }
    }
}

In [38]:
# =========================
# 6D) CALL MODEL FOR AGGREGATION ONLY
# =========================

# def call_model_for_aggregation_only(subset_payload: dict, system_prompt: str, policy_context: str) -> List[dict]:
#     max_attempts = 5
#     sleep_s = 2

#     for attempt in range(1, max_attempts + 1):
#         try:
#             resp = client.chat.completions.create(
#                 model="gpt-5.1",
#                 messages=[
#                     {"role": "system", "content": system_prompt},
#                     {"role": "user", "content": "Here is the procurement plan data as JSON.\n\n" + json.dumps(subset_payload)},
#                     {"role": "user", "content": "Regulations excerpts:\n\n" + policy_context},
#                 ],
#                 response_format=response_format_agg_only,
#                 temperature=0
#             )
#             raw = resp.choices[0].message.content
#             obj = json.loads(raw)
#             return obj.get("improvement_table", []) or []

#         except Exception as e:
#             if attempt == max_attempts:
#                 print(f"❌ LLM call failed after {max_attempts} attempts: {e}")
#                 return []
#             print(f"⚠️ LLM call attempt {attempt}/{max_attempts} failed: {e} — retrying in {sleep_s}s")
#             time.sleep(sleep_s)
#             sleep_s = min(sleep_s * 2, 30)

def call_model_for_aggregation_only(subset_payload: dict, system_prompt: str, policy_context: str) -> List[dict]:
    resp = client.chat.completions.create(
        model="gpt-5.1",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "Here is the procurement plan data as JSON.\n\n" + json.dumps(subset_payload)},
            {"role": "user", "content": "Regulations excerpts:\n\n" + policy_context},
        ],
        response_format=response_format_agg_only,
        temperature=0
    )
    raw = resp.choices[0].message.content
    obj = json.loads(raw)
    return obj.get("improvement_table", []) or []


def call_model_for_aggregation_batch(
    batch_payload: dict,
    system_prompt: str,
    policy_context: str
) -> List[dict]:
    """
    batch_payload contains:
      - project_id
      - user_type
      - activities (all activities referenced by the batch groups)
      - candidate_groups: list of lists of activity_ids
    Model should output improvement_table rows for any groups it recommends.
    """
    instructions = (
        "You will receive candidate_groups: a list of candidate sets of activity_ids.\n"
        "For EACH candidate group, decide whether aggregation is recommended.\n"
        "If recommended, output exactly ONE improvement_table row for that group.\n"
        "The row's 'Activity IDs' MUST be exactly the comma-separated ids from that group (no extras, no missing).\n"
        "If NOT recommended, output nothing for that group.\n"
        "Do not merge groups. Treat each candidate group independently.\n"
    )

    resp = client.chat.completions.create(
        model="gpt-5.1",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": instructions + "\n\nBatch payload:\n\n" + json.dumps(batch_payload)},
            {"role": "user", "content": "Regulations excerpts:\n\n" + policy_context},
        ],
        response_format=response_format_agg_only,
        temperature=0
    )
    raw = resp.choices[0].message.content
    obj = json.loads(raw)
    return obj.get("improvement_table", []) or []



In [39]:
# =========================
# 6E) HELPERS FOR BATCHING
# =========================

def chunk_list(lst: List, n: int) -> List[List]:
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


In [40]:
# =========================
# 7) AGGREGATION ONLY (BATCHED)
# =========================
print("Project", TARGET_PROJ_ID)
print("df_project rows:", len(df_project))
print("activities_payload length:", len(activities_payload))

policy_context = retrieve_policy_chunks_for_plan(user_payload, top_k=15)

# Exclude INDV early
activities_for_candidates = [a for a in activities_payload if (a.get("method") or "").strip().upper() != "INDV"]

# Build candidate groups
# cat_groups = build_activity_candidates(
#     activities_for_candidates,
#     embed_fn=embed_texts,
#     k=15,                   # 20
#     sim_threshold=0.42,     # 0.35
#     max_desc_chars=300,
#     max_groups_per_cat=20,
#     max_group_size=20
# )
cat_groups = build_activity_candidates(
    activities_for_candidates,
    embed_fn=embed_texts,
    k=15,
    sim_threshold=0.42,
    max_desc_chars=300,
    strategy="top_clusters",
    max_groups_per_cat=20,
    max_group_size=20,
    top_pairs_per_cat=250
)


# Flatten + dedupe candidate groups
seen = set()
candidate_groups = []
for _, groups in cat_groups.items():
    for g in groups:
        key = tuple(sorted(g))
        if key in seen:
            continue
        seen.add(key)
        candidate_groups.append(list(g))

print("Candidate groups:", len(candidate_groups))

# Call model per candidate group (small payloads)
# merged_improvement = []
# for i, group_ids in enumerate(candidate_groups, start=1):
#     gid_set = set(group_ids)
#     subset_activities = [a for a in activities_for_candidates if a["activity_id"] in gid_set]

#     if len(subset_activities) < 2:
#         continue

#     subset_payload = {
#         "user_type": "Borrower",
#         "project_id": TARGET_PROJ_ID,
#         "activities": subset_activities
#     }

#     print(f"LLM aggregation batch {i}/{len(candidate_groups)} (n={len(subset_activities)})")
#     short_policy_context = policy_context[:4000]
#     rows = call_model_for_aggregation_only(subset_payload, system_prompt_agg_ag, policy_context)
#     merged_improvement.extend(rows)
# Call model per BATCH of candidate groups (much fewer calls)
merged_improvement = []

GROUP_BATCH_SIZE = 8  # tune: 5-15 is typical; higher = fewer calls but bigger payload

short_policy_context = policy_context[:4000]  # optional but recommended for scale

for b, group_batch in enumerate(chunk_list(candidate_groups, GROUP_BATCH_SIZE), start=1):
    # Collect all activity_ids referenced by this batch
    batch_ids = set()
    for g in group_batch:
        batch_ids.update(g)

    batch_activities = [a for a in activities_for_candidates if a["activity_id"] in batch_ids]

    # Skip degenerate batches
    if len(batch_activities) < 2:
        continue

    batch_payload = {
        "user_type": "Borrower",
        "project_id": TARGET_PROJ_ID,
        "activities": batch_activities,
        "candidate_groups": group_batch
    }

    print(f"LLM group-batch {b} (groups={len(group_batch)} | activities={len(batch_activities)})")
    rows = call_model_for_aggregation_batch(batch_payload, system_prompt_agg_ag, short_policy_context)
    merged_improvement.extend(rows)


# Deduplicate improvement rows by normalized Activity IDs set
dedup = {}
for row in merged_improvement:
    ids_raw = row.get("Activity IDs", "") or ""
    ids = tuple(sorted(i.strip() for i in ids_raw.split(",") if i.strip()))
    if not ids:
        continue
    dedup[ids] = row
merged_improvement = list(dedup.values())

# -------------------------
# Build deterministic plan_review (so later sections never crash)
# -------------------------
modified_plan = []
for a in activities_payload:
    modified_plan.append({
        "activity_id": a["activity_id"],
        "original_start_date": a.get("planned_dates", {}).get("activity_start_date", "") or "",
        "original_end_date": a.get("planned_dates", {}).get("activity_end_date", "") or "",
        "original_estimated_cost_usd": float(a.get("estimated_cost_usd", 0.0) or 0.0),
        "recommended_start_date": a.get("planned_dates", {}).get("activity_start_date", "") or "",
        "recommended_end_date": a.get("planned_dates", {}).get("activity_end_date", "") or "",
        "recommended_estimated_cost_usd": float(a.get("estimated_cost_usd", 0.0) or 0.0),
        "aggregation_group": None,
        "compliance_status": "compliant",
        "key_changes": ""
    })

scored_dashboard = []
for a in activities_payload:
    scored_dashboard.append({
        "activity_id": a["activity_id"],
        "activity_name": a.get("activity_name", ""),
        "category": a.get("procurement category", ""),
        "method": a.get("method", ""),
        "value_usd_million": float(a.get("estimated_cost_usd", 0.0) or 0.0) / 1_000_000.0,
        "risk_planning": 0,
        "risk_market": 0,
        "risk_cost": 0,
        "risk_timelines": 0,
        "risk_dependency": 0,
        "risk_aggregation": 0,
        "composite_score": 0,
        "top_flags": []
    })

plan_review = {
    "executive_summary": {
        "total_activities": canonical_activity_count,
        "high_risk": 0,
        "medium_risk": 0,
        "low_risk": canonical_activity_count,
        "top_recommendations": [],
        "expected_impact": ""
    },
    "findings_by_class": [],
    "scored_dashboard": scored_dashboard,
    "policy_alignment": [],
    "improvement_table": merged_improvement,
    "modified_plan": modified_plan
}

Project P166309
df_project rows: 345
activities_payload length: 345
To sign in, use a web browser to open the page https://microsoft.com/devicelogin and enter the code BPCV24B26 to authenticate.
Candidate groups: 32
LLM group-batch 1 (groups=8 | activities=60)
LLM group-batch 2 (groups=8 | activities=63)
LLM group-batch 3 (groups=8 | activities=40)
LLM group-batch 4 (groups=8 | activities=20)


In [41]:
# THE SECTION ABOVE IS DIFFERENT COMPARED TO AGGREGATION AGENT 18

# modified plan, scored_dashboard, and plan review are all either added, rearranged, or modified compared to Aggregation Agent 18.

In [42]:
# =========================
# 7B) ENFORCE COUNTS
# =========================
xs = plan_review.get("executive_summary", {}) or {}

# Force the model’s summary to align with actual objects
n_scored = len(plan_review.get("scored_dashboard", []))
n_mod    = len(plan_review.get("modified_plan", []))
n_canon  = len(canonical_activity_ids)

# Option: assert and fail fast if something is wildly off:
if n_scored not in (0, n_canon) or n_mod not in (0, n_canon):
    print("⚠️ Inconsistent counts:",
          "canonical=", n_canon,
          "scored_dashboard=", n_scored,
          "modified_plan=", n_mod)

# Make total_activities deterministic instead of trusting the model
xs["total_activities"] = n_canon
plan_review["executive_summary"] = xs

In [43]:
# =========================
# 8) OUTPUTS (Console + Excel for Improvement Table)
# =========================
print("\nProcurement Plan Review (Project:", TARGET_PROJ_ID, ")")
xs = plan_review.get("executive_summary", {})
print(f"Total current activities: {xs.get('total_activities')}")
print(f"High risk: {xs.get('high_risk')} | Medium risk: {xs.get('medium_risk')} | Low risk: {xs.get('low_risk')}")
print("Top recommendations:")
for i, rec in enumerate(xs.get("top_recommendations", [])[:5], start=1):
    print(f"  {i}. {rec}")
print("Expected impact:", xs.get("expected_impact", ""))

# Map ACTVTY_ID -> cost from df_project
id_to_cost = {
    str(r["ACTVTY_ID"]): float(r["EST_BUDG_AMT"]) if pd.notna(r["EST_BUDG_AMT"]) else 0.0
    for _, r in df_project.iterrows()
}

improvement_table = plan_review.get("improvement_table", []) or []

cleaned_rows = []
for row in improvement_table:
    ids_raw = row.get("Activity IDs", "") or ""
    ids = [i.strip() for i in ids_raw.split(",") if i.strip()]

    row["Number of Activities"] = len(ids)

    sum_cost = sum(id_to_cost.get(i, 0.0) for i in ids)
    row["Sum of Costs"] = f"{sum_cost:.2f}"

    cleaned_rows.append(row)

improvement_table = cleaned_rows
plan_review["improvement_table"] = improvement_table

if improvement_table:
    df_table = pd.DataFrame(improvement_table)
    out_path = f"C:/Users/wb643925/OneDrive - WBG/SmartProcurement/Aggregation_Table_{TARGET_PROJ_ID}.xlsx"
    df_table.to_excel(out_path, index=False)
    print(f"✅ Improvement Table saved to {out_path}")

scored_ids   = {row.get("activity_id") for row in plan_review.get("scored_dashboard", [])}
modified_ids = {row.get("activity_id") for row in plan_review.get("modified_plan", [])}
canonical_ids = set(canonical_activity_ids)

if scored_ids and scored_ids != canonical_ids:
    print("⚠️ scored_dashboard IDs != canonical IDs")
    print("Missing from scored_dashboard:", canonical_ids - scored_ids)
    print("Extra in scored_dashboard:", scored_ids - canonical_ids)

if modified_ids and modified_ids != canonical_ids:
    print("⚠️ modified_plan IDs != canonical IDs")
    print("Missing from modified_plan:", canonical_ids - modified_ids)
    print("Extra in modified_plan:", modified_ids - canonical_ids)



Procurement Plan Review (Project: P166309 )
Total current activities: 345
High risk: 0 | Medium risk: 0 | Low risk: 345
Top recommendations:
Expected impact: 
✅ Improvement Table saved to C:/Users/wb643925/OneDrive - WBG/SmartProcurement/Aggregation_Table_P166309.xlsx


In [44]:
# THE SECTION ABOVE IS DIFFERENT COMPARED TO AGGREGATION AGENT 18

# Wait, why is improvement table first set equal to plan_riview["improvement_table"],
# then why is it set equal to cleaned_rows (for cleaning?),
# and then why is plan_review["improvement_table"] set equal to improvement_table again?


In [45]:

# just run code, see if it crashes (the batch sizing still needs to be iterated on)
## XX if it runs, tell chatgpt that the code worked, but ask about the differences
### XX especially with the prompt, JSON response format, and the improvement table handling

## if the code doesn't run, try with a smaller project
### if it runs with a smaller project, tell chatgpt that the code worked, but ask about the differences (generated report)

# revise prompt
## does anything from the old prompt need to be brought back?
## how do we incorporate the new rules?
